In [ ]:
# ============================================================
#  环境配置
#  - Colab 用户：取消注释下方 Colab 区块
#  - 本地 Jupyter 用户：默认仅在缺少依赖时安装
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install torch torchvision transformers pillow matplotlib numpy -q

# ── 本地 Jupyter 环境 ──
import subprocess, sys, importlib.util

def _ensure_pkg(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or import_name, "-q"])

_ensure_pkg("torch")
_ensure_pkg("torchvision")
_ensure_pkg("transformers")
_ensure_pkg("PIL", "pillow")
_ensure_pkg("matplotlib")
_ensure_pkg("numpy")

In [ ]:
import math, random, os
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ViLT 实战：架构解析 + 预训练推理

基于论文 *ViLT: Vision-and-Language Transformer Without Convolution or Region Supervision* (Kim et al., 2021)，
用 **图文检索 / 图文匹配打分** 任务演示 ViLT 的核心架构与推理流程。

| | 架构解析 | 预训练推理 |
|---|---|---|
| 核心思路 | 逐组件拆解 ViLT 关键模块，含公式推导 | 加载检索任务已微调权重，直接做图文匹配打分 |
| 重点 | 理解 Patch Projection 替代 CNN 的设计动机 | 掌握 HuggingFace `ViltForImageAndTextRetrieval` 的正确用法 |
| 适合场景 | 面试准备、多模态原理深入 | 工程推理、快速验证 |

## 1. 数据准备

我们构造一个轻量级的 **图文匹配 (ITM)** 数据集：
- 用 PIL 程序化生成彩色几何图形图片（`224×224`）
- 为每张图片构造匹配文本和不匹配文本
- 二分类任务：label = 1（匹配） / 0（不匹配）

该数据集无需外部下载，可直接在 CPU 上高效运行。

In [ ]:
# ── 程序化生成 ITM 数据集 ──

COLORS = {
    'red': (220, 50, 50), 'blue': (50, 50, 220), 'green': (50, 180, 50),
    'yellow': (220, 220, 50), 'purple': (150, 50, 200), 'orange': (240, 150, 30),
}
SHAPES = ['rectangle', 'circle', 'triangle']
IMG_SIZE = 224


def generate_shape_image(color_name: str, shape_name: str) -> Image.Image:
    """生成一张包含指定颜色和形状的 PIL 图片."""
    img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (245, 245, 245))
    draw = ImageDraw.Draw(img)
    rgb = COLORS[color_name]
    # 随机偏移位置，增加多样性
    cx, cy = IMG_SIZE // 2 + random.randint(-20, 20), IMG_SIZE // 2 + random.randint(-20, 20)
    r = random.randint(40, 60)
    if shape_name == 'rectangle':
        draw.rectangle([cx - r, cy - r, cx + r, cy + r], fill=rgb)
    elif shape_name == 'circle':
        draw.ellipse([cx - r, cy - r, cx + r, cy + r], fill=rgb)
    elif shape_name == 'triangle':
        draw.polygon([(cx, cy - r), (cx - r, cy + r), (cx + r, cy + r)], fill=rgb)
    return img


def build_itm_dataset(n_per_combo: int = 6):
    """构造 ITM 数据集: (image, text, label).
    每个 (颜色, 形状) 组合生成 n_per_combo 张图，各配一条匹配文本和一条不匹配文本.
    """
    images, texts, labels = [], [], []
    color_names = list(COLORS.keys())
    combos = [(c, s) for c in color_names for s in SHAPES]  # 18 种组合

    for color, shape in combos:
        for _ in range(n_per_combo):
            img = generate_shape_image(color, shape)
            correct_text = f"a {color} {shape}"

            # 正样本
            images.append(img); texts.append(correct_text); labels.append(1)

            # 负样本：随机选一个不同的描述
            wrong_color = random.choice([c for c in color_names if c != color])
            wrong_shape = random.choice([s for s in SHAPES if s != shape])
            wrong_text = f"a {wrong_color} {wrong_shape}"
            images.append(img.copy()); texts.append(wrong_text); labels.append(0)

    return images, texts, labels


all_images, all_texts, all_labels = build_itm_dataset(n_per_combo=6)
print(f"Total samples: {len(all_images)} (pos={sum(all_labels)}, neg={len(all_labels)-sum(all_labels)})")

In [ ]:
# ── 可视化部分样本 ──
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
indices = random.sample(range(len(all_images)), 8)
for ax, idx in zip(axes.flat, indices):
    ax.imshow(all_images[idx])
    lbl = "MATCH" if all_labels[idx] == 1 else "MISMATCH"
    ax.set_title(f'"{all_texts[idx]}" [{lbl}]', fontsize=9)
    ax.axis('off')
plt.suptitle("Sample ITM Data", fontsize=13)
plt.tight_layout()
plt.show()

## 2. 超参数

架构解析使用缩小版参数（Mini-ViLT），用于演示组件结构和张量流动；应用部分使用 HuggingFace 已在检索任务上微调好的 ViLT 权重，只做推理评估。

In [ ]:
# ── Mini-ViLT 超参数（架构解析用，缩小以适配 CPU） ──
MINI_D_MODEL    = 64      # 模型维度（论文 768）
MINI_NUM_HEADS  = 4       # 注意力头数（论文 12）
MINI_NUM_LAYERS = 2       # Transformer 层数（论文 12）
MINI_D_FF       = 128     # FFN 隐藏维度（论文 3072）
MINI_PATCH_SIZE = 32      # Patch 大小（论文 32）
MINI_IMG_SIZE   = 64      # 图像尺寸（论文 384）
MINI_VOCAB_SIZE = 1000    # 词表大小（论文 30522）
MINI_MAX_SEQ    = 40      # 最大序列长度
MINI_DROPOUT    = 0.1

# ── 推理超参数 ──
EVAL_BATCH_SIZE = 8
MATCH_THRESHOLD = 0.0   # logit > 0 等价于 sigmoid(logit) > 0.5

## 3. 架构解析 — 逐组件拆解 ViLT

ViLT 的核心思想：**将视觉输入的处理简化到与文本一致**。图像不再经过 CNN 或目标检测器，而是直接切分为 Patch 后做线性映射，与文本 token 拼接送入统一的 Transformer Encoder。

下面逐组件实现 Mini-ViLT，每个组件配有公式推导和张量形状标注。

### 3.1 Patch Embedding（视觉嵌入）

ViLT 借鉴 ViT 的思想，将图像切分为不重叠的 Patch，然后通过一个线性投影映射到模型维度：

$$v_i = \text{Linear}(\text{flatten}(\text{patch}_i)) \in \mathbb{R}^{d}, \quad i = 1, \ldots, N$$

其中 $N = (H/P) \times (W/P)$ 是 Patch 数量，$P$ 为 Patch 大小。

与传统方法的对比：
- **Region Feature (Faster R-CNN)**：耗时 ~810ms，需要 RPN + RoI + NMS
- **Grid Feature (ResNet)**：耗时 ~45ms，需要完整 CNN 前向传播
- **Patch Projection (ViLT)**：耗时 ~0.4ms，仅需一个线性层

实现上用 `nn.Conv2d(kernel_size=P, stride=P)` 等价于将每个 patch 展平后做线性映射。

In [ ]:
class PatchEmbedding(nn.Module):
    """将图像切分为 patch 并线性投影到 d_model 维度.

    等价操作: flatten(patch) @ W + b, 用 Conv2d(kernel=P, stride=P) 高效实现.
    """
    def __init__(self, img_size: int, patch_size: int, d_model: int, in_channels: int = 3):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        # Conv2d(kernel=P, stride=P) 等价于对每个 patch 做线性映射
        self.proj = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)
        # 可学习的 [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        # 位置嵌入: [CLS] + N patches
        self.pos_embed = nn.Parameter(torch.randn(1, 1 + self.num_patches, d_model))

    def forward(self, x):
        # x: (batch, 3, H, W)
        B = x.shape[0]
        x = self.proj(x)              # (batch, d_model, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)  # (batch, num_patches, d_model)
        cls = self.cls_token.expand(B, -1, -1)  # (batch, 1, d_model)
        x = torch.cat([cls, x], dim=1)    # (batch, 1+num_patches, d_model)
        x = x + self.pos_embed            # (batch, 1+num_patches, d_model)
        return x


# ── 验证 ──
patch_emb = PatchEmbedding(MINI_IMG_SIZE, MINI_PATCH_SIZE, MINI_D_MODEL)
dummy_img = torch.randn(2, 3, MINI_IMG_SIZE, MINI_IMG_SIZE)  # batch=2
v = patch_emb(dummy_img)
num_patches = (MINI_IMG_SIZE // MINI_PATCH_SIZE) ** 2
print(f"Input:  {tuple(dummy_img.shape)}")
print(f"Output: {tuple(v.shape)}  (expected: (2, {1+num_patches}, {MINI_D_MODEL}))")

### 3.2 Text Embedding（文本嵌入）

文本处理与 BERT 一致：

$$t_i = \text{WordEmbed}(\text{token}_i) + \text{PosEmbed}(i) \in \mathbb{R}^{d}$$

文本序列同样在开头加入一个可学习的 `[CLS]` token，最终序列长度为 $L + 1$（$L$ 为文本 token 数）。

In [ ]:
class TextEmbedding(nn.Module):
    """文本嵌入 = 词嵌入 + 位置嵌入, 序列开头加 [CLS] token."""
    def __init__(self, vocab_size: int, d_model: int, max_seq_len: int):
        super().__init__()
        self.word_embed = nn.Embedding(vocab_size, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        # 位置嵌入: [CLS] + max_seq_len tokens
        self.pos_embed = nn.Parameter(torch.randn(1, 1 + max_seq_len, d_model))

    def forward(self, input_ids):
        # input_ids: (batch, seq_len) — 整数索引
        B, L = input_ids.shape
        x = self.word_embed(input_ids)         # (batch, seq_len, d_model)
        cls = self.cls_token.expand(B, -1, -1) # (batch, 1, d_model)
        x = torch.cat([cls, x], dim=1)         # (batch, 1+seq_len, d_model)
        x = x + self.pos_embed[:, :1+L, :]     # (batch, 1+seq_len, d_model)
        return x


# ── 验证 ──
text_emb = TextEmbedding(MINI_VOCAB_SIZE, MINI_D_MODEL, MINI_MAX_SEQ)
dummy_ids = torch.randint(0, MINI_VOCAB_SIZE, (2, 10))  # batch=2, seq_len=10
t = text_emb(dummy_ids)
print(f"Input:  {tuple(dummy_ids.shape)}")
print(f"Output: {tuple(t.shape)}  (expected: (2, 11, {MINI_D_MODEL}))")

### 3.3 模态类型嵌入与多模态拼接

ViLT 的关键设计：两个模态的嵌入各自加上**模态类型嵌入 (Modal Type Embedding)**，然后在序列维度拼接，统一送入同一个 Transformer：

$$z^0 = [\bar{t} + t^{\text{type}};\; \bar{v} + v^{\text{type}}]$$

- $t^{\text{type}}$: 文本模态的类型嵌入（type_id = 0）
- $v^{\text{type}}$: 视觉模态的类型嵌入（type_id = 1）

这种**单流融合 (Single-Stream)** 设计让文本和视觉 token 从第一层就能通过 Self-Attention 相互交互，而非像双流模型那样先分别编码再融合。

In [ ]:
class MultimodalFusion(nn.Module):
    """给两个模态分别加上类型嵌入, 然后在序列维度拼接."""
    def __init__(self, d_model: int, num_modalities: int = 2):
        super().__init__()
        self.type_embed = nn.Embedding(num_modalities, d_model)

    def forward(self, text_emb, visual_emb):
        # text_emb:   (batch, L_t, d_model) — 包含 [CLS_t]
        # visual_emb: (batch, L_v, d_model) — 包含 [CLS_v]
        B = text_emb.shape[0]
        L_t, L_v = text_emb.shape[1], visual_emb.shape[1]

        text_type = self.type_embed(torch.zeros(B, L_t, dtype=torch.long, device=text_emb.device))
        vis_type  = self.type_embed(torch.ones(B, L_v, dtype=torch.long, device=visual_emb.device))

        text_emb   = text_emb   + text_type   # (batch, L_t, d_model)
        visual_emb = visual_emb + vis_type     # (batch, L_v, d_model)

        z = torch.cat([text_emb, visual_emb], dim=1)  # (batch, L_t + L_v, d_model)
        return z


# ── 验证 ──
fusion = MultimodalFusion(MINI_D_MODEL)
z0 = fusion(t, v)
print(f"Text:   {tuple(t.shape)}")
print(f"Visual: {tuple(v.shape)}")
print(f"Fused:  {tuple(z0.shape)}  (expected: (2, {t.shape[1]+v.shape[1]}, {MINI_D_MODEL}))")

### 3.4 Pre-norm Transformer Encoder

ViLT 使用 ViT 风格的 **Pre-norm** 结构（先 LayerNorm 再做 Attention/FFN），区别于 BERT 的 Post-norm：

$$\hat{z}^d = \text{MSA}(\text{LN}(z^{d-1})) + z^{d-1}$$

$$z^d = \text{MLP}(\text{LN}(\hat{z}^d)) + \hat{z}^d$$

Pre-norm 的优势：梯度更稳定，训练更容易收敛，尤其对深层 Transformer 至关重要。

In [ ]:
class PreNormTransformerLayer(nn.Module):
    """Pre-norm Transformer Encoder Layer (ViT 风格).

    与 Post-norm (BERT 风格) 的区别:
      Pre-norm:  x -> LN -> Attn -> + residual -> LN -> FFN -> + residual
      Post-norm: x -> Attn -> + residual -> LN -> FFN -> + residual -> LN
    """
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, attn_mask=None):
        # x: (batch, seq_len, d_model)
        # ── MSA block (pre-norm) ──
        x_norm = self.ln1(x)                             # (batch, seq, d_model)
        attn_out, attn_weights = self.attn(
            x_norm, x_norm, x_norm, attn_mask=attn_mask
        )                                                # (batch, seq, d_model)
        x = x + attn_out                                 # residual

        # ── FFN block (pre-norm) ──
        x_norm = self.ln2(x)                             # (batch, seq, d_model)
        x = x + self.ffn(x_norm)                         # residual
        return x, attn_weights


# ── 验证 ──
layer = PreNormTransformerLayer(MINI_D_MODEL, MINI_NUM_HEADS, MINI_D_FF)
z1, weights = layer(z0)
print(f"Input:  {tuple(z0.shape)}")
print(f"Output: {tuple(z1.shape)}  (shape preserved)")
print(f"Attn weights: {tuple(weights.shape)}")  # (batch, seq, seq)

### 3.5 池化表示与任务头

ViLT 使用文本 `[CLS]` token（序列第一个 token）的输出作为池化表示：

$$p = \tanh(z_0^D \cdot W_{\text{pool}})$$

在此基础上构建两个预训练任务头：

- **ITM (Image-Text Matching)**：二分类线性层，预测图文是否匹配
- **MLM (Masked Language Modeling)**：2 层 MLP，预测被 mask 的文本 token

In [ ]:
class Pooler(nn.Module):
    """取第一个 token ([CLS_t]) 的输出，经 tanh 激活得到池化表示."""
    def __init__(self, d_model: int):
        super().__init__()
        self.dense = nn.Linear(d_model, d_model)

    def forward(self, hidden_states):
        # hidden_states: (batch, seq_len, d_model)
        cls_output = hidden_states[:, 0]    # (batch, d_model) — 取第一个 token
        pooled = torch.tanh(self.dense(cls_output))  # (batch, d_model)
        return pooled


class ITMHead(nn.Module):
    """Image-Text Matching head: 二分类."""
    def __init__(self, d_model: int):
        super().__init__()
        self.classifier = nn.Linear(d_model, 2)

    def forward(self, pooled):
        # pooled: (batch, d_model)
        return self.classifier(pooled)  # (batch, 2)


class MLMHead(nn.Module):
    """Masked Language Modeling head: 2 层 MLP 预测原始 token."""
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
            nn.Linear(d_model, vocab_size),
        )

    def forward(self, hidden_states):
        # hidden_states: (batch, seq_len, d_model) — 只取被 mask 的位置
        return self.mlp(hidden_states)  # (batch, seq_len, vocab_size)


# ── 验证 ──
pooler = Pooler(MINI_D_MODEL)
itm_head = ITMHead(MINI_D_MODEL)
mlm_head = MLMHead(MINI_D_MODEL, MINI_VOCAB_SIZE)

p = pooler(z1)
itm_logits = itm_head(p)
mlm_logits = mlm_head(z1[:, :t.shape[1], :])  # 只取文本部分

print(f"Pooled:     {tuple(p.shape)}")
print(f"ITM logits: {tuple(itm_logits.shape)}  (batch, 2)")
print(f"MLM logits: {tuple(mlm_logits.shape)}  (batch, text_len, vocab_size)")

### 3.6 完整 Mini-ViLT 组装

将上述组件组装成完整的 Mini-ViLT 模型，验证端到端数据流：

```
Image (B,3,H,W)  →  PatchEmbedding  →  (B, 1+N, d)  ─┐
                                                        ├→ MultimodalFusion → (B, L_t+L_v, d) → Transformer×D → Pooler → ITM/MLM
Text  (B, L)     →  TextEmbedding   →  (B, 1+L, d)  ─┘
```

In [ ]:
class MiniViLT(nn.Module):
    """缩小版 ViLT，用于演示完整数据流."""
    def __init__(self, img_size, patch_size, vocab_size, d_model,
                 num_heads, num_layers, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, d_model)
        self.text_embed  = TextEmbedding(vocab_size, d_model, max_seq_len)
        self.fusion      = MultimodalFusion(d_model)
        self.encoder = nn.ModuleList([
            PreNormTransformerLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)
        self.pooler   = Pooler(d_model)
        self.itm_head = ITMHead(d_model)
        self.text_seq_offset = 0  # 记录文本在拼接序列中的长度

    def forward(self, images, input_ids, labels=None):
        # images:    (batch, 3, H, W)
        # input_ids: (batch, seq_len)
        v = self.patch_embed(images)       # (batch, 1+N, d_model)
        t = self.text_embed(input_ids)     # (batch, 1+L, d_model)
        self.text_seq_offset = t.shape[1]

        z = self.fusion(t, v)              # (batch, (1+L)+(1+N), d_model)

        attn_weights_all = []
        for layer in self.encoder:
            z, w = layer(z)                # (batch, total_seq, d_model)
            attn_weights_all.append(w)

        z = self.final_ln(z)               # (batch, total_seq, d_model)
        pooled = self.pooler(z)            # (batch, d_model)
        itm_logits = self.itm_head(pooled) # (batch, 2)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(itm_logits, labels)

        return {'loss': loss, 'logits': itm_logits, 'attn_weights': attn_weights_all}


# ── 端到端验证 ──
mini_vilt = MiniViLT(
    MINI_IMG_SIZE, MINI_PATCH_SIZE, MINI_VOCAB_SIZE, MINI_D_MODEL,
    MINI_NUM_HEADS, MINI_NUM_LAYERS, MINI_D_FF, MINI_MAX_SEQ, MINI_DROPOUT
)
out = mini_vilt(dummy_img, dummy_ids, labels=torch.tensor([0, 1]))

print(f"Model parameters: {sum(p.numel() for p in mini_vilt.parameters()):,}")
print(f"Loss:   {out['loss'].item():.4f}")
print(f"Logits: {tuple(out['logits'].shape)}")
print(f"Attn layers: {len(out['attn_weights'])}, each: {tuple(out['attn_weights'][0].shape)}")

## 4. 预训练推理 — HuggingFace `ViltForImageAndTextRetrieval`

上一节我们从零实现了 ViLT 的核心组件。对 ViLT 这类大模型，更适合在教学 Notebook 中采用 **策略 C：预训练推理 / 应用**。

**为什么这里不用微调，而改为推理演示？**
- `dandelin/vilt-b32-mlm` 是 MLM checkpoint，不是检索任务 checkpoint
- `ViltForImageAndTextRetrieval` 的当前官方文档说明 `labels` **not supported**，不适合直接沿用原 notebook 的 `outputs.loss` 训练写法
- 在 CPU 教学环境中，直接使用已在 COCO 检索任务上微调好的权重更稳妥，也更符合“开箱即跑”的目标

**推理流程**：
1. 用 `ViltProcessor` 同时处理图像与文本
2. 用 `ViltForImageAndTextRetrieval` 输出单个匹配分数 `logits`
3. 用 `sigmoid(logit)` 转成匹配概率，观察 matched / mismatched 文本的差异

| 对应关系 | 架构解析 (Mini-ViLT) | 预训练推理 (HuggingFace) |
|---------|---------------------|------------------------|
| Patch Embedding | `PatchEmbedding` | `ViltModel.embeddings.patch_embeddings` |
| Text Embedding | `TextEmbedding` | `ViltModel.embeddings.text_embeddings` |
| Type Embedding | `MultimodalFusion.type_embed` | `ViltModel.embeddings.token_type_embeddings` |
| Transformer | `PreNormTransformerLayer` | `ViltModel.encoder.layer[i]` |
| Retrieval Head | `ITMHead`（教学版二分类） | `ViltForImageAndTextRetrieval.rank_output`（单 logit 打分） |

In [ ]:
from transformers import ViltProcessor, ViltForImageAndTextRetrieval

# ── 加载检索任务已微调好的模型和处理器 ──
model_name = "dandelin/vilt-b32-finetuned-coco"
processor = ViltProcessor.from_pretrained(model_name)
model = ViltForImageAndTextRetrieval.from_pretrained(model_name)
model.to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Checkpoint: {model_name}")
print(f"Total parameters: {total_params:,}")
print(f"Model architecture: {model.__class__.__name__}")

In [ ]:
# ── 检查检索头与输出语义 ──
print(f"Has rank_output head: {hasattr(model, 'rank_output')}")
if hasattr(model, 'rank_output'):
    head_params = sum(p.numel() for p in model.rank_output.parameters())
    print(f"rank_output params: {head_params:,}")

print("This model returns one retrieval logit per image-text pair.")
print("Interpretation: larger logit => more likely to match.")

In [ ]:
# ── 用 ViltProcessor 预处理全部数据 ──

def preprocess_all(images, texts, labels, processor, batch_size=32):
    """分批调用 processor 预处理，避免一次性占用过多内存."""
    all_input_ids, all_attention_mask = [], []
    all_token_type_ids, all_pixel_values, all_pixel_mask = [], [], []

    for i in range(0, len(images), batch_size):
        batch_imgs  = images[i:i+batch_size]
        batch_texts = texts[i:i+batch_size]
        enc = processor(
            images=batch_imgs, text=batch_texts,
            return_tensors="pt", padding="max_length",
            truncation=True, max_length=40
        )
        all_input_ids.append(enc['input_ids'])
        all_attention_mask.append(enc['attention_mask'])
        all_token_type_ids.append(enc['token_type_ids'])
        all_pixel_values.append(enc['pixel_values'])
        all_pixel_mask.append(enc['pixel_mask'])

    return {
        'input_ids':      torch.cat(all_input_ids),
        'attention_mask':  torch.cat(all_attention_mask),
        'token_type_ids': torch.cat(all_token_type_ids),
        'pixel_values':   torch.cat(all_pixel_values),
        'pixel_mask':     torch.cat(all_pixel_mask),
        'labels':         torch.tensor(labels, dtype=torch.long),
    }


encoded = preprocess_all(all_images, all_texts, all_labels, processor)

# 划分训练集和测试集 (80/20)
n_total = len(all_labels)
n_train = int(n_total * 0.8)
indices = torch.randperm(n_total)
train_idx, test_idx = indices[:n_train], indices[n_train:]

print(f"Train: {len(train_idx)} | Test: {len(test_idx)}")
print(f"pixel_values shape: {tuple(encoded['pixel_values'].shape)}")
print(f"input_ids shape:    {tuple(encoded['input_ids'].shape)}")

In [ ]:
class ITMDataset(Dataset):
    """从预处理好的 encoded dict 中按索引取样."""
    def __init__(self, encoded_data, indices):
        self.data = encoded_data
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        return {k: v[i] for k, v in self.data.items()}


train_dataset = ITMDataset(encoded, train_idx)
test_dataset  = ITMDataset(encoded, test_idx)
train_loader  = DataLoader(train_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False)
test_loader   = DataLoader(test_dataset,  batch_size=EVAL_BATCH_SIZE)

# 验证一个 batch
batch = next(iter(test_loader))
for k, v in batch.items():
    print(f"  {k}: {tuple(v.shape)}")

In [ ]:
# ── 批量推理：在测试集上计算匹配分数 ──
model.eval()
test_logits_list, test_labels_list = [], []

with torch.no_grad():
    for batch in test_loader:
        labels = batch['labels']
        model_inputs = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
        outputs = model(**model_inputs)
        logits = outputs.logits.squeeze(-1).cpu()   # (batch,)

        test_logits_list.append(logits)
        test_labels_list.append(labels.cpu())

test_logits = torch.cat(test_logits_list)
test_labels = torch.cat(test_labels_list)
test_probs = torch.sigmoid(test_logits)
test_preds = (test_logits > MATCH_THRESHOLD).long()

print(f"Collected logits: {tuple(test_logits.shape)}")
print(f"Mean prob (match labels=1): {test_probs[test_labels == 1].mean().item():.3f}")
print(f"Mean prob (match labels=0): {test_probs[test_labels == 0].mean().item():.3f}")

In [ ]:
# ── 测试集评估 ──
accuracy = (test_preds == test_labels).float().mean().item()
bce = F.binary_cross_entropy_with_logits(test_logits, test_labels.float()).item()

print(f"Heuristic Accuracy (threshold=0): {accuracy:.2%}")
print(f"Binary Cross-Entropy: {bce:.4f}")
print("Note: threshold 0 is a simple teaching heuristic, not a calibrated retrieval decision rule.")

In [ ]:
# ── 推理演示：对几个样本进行图文匹配打分 ──
model.eval()
demo_indices = random.sample(range(len(test_idx)), min(6, len(test_idx)))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, di in zip(axes.flat, demo_indices):
    real_idx = test_idx[di].item()
    img = all_images[real_idx]
    text = all_texts[real_idx]
    true_label = all_labels[real_idx]

    inputs = processor(images=img, text=text, return_tensors="pt").to(device)
    with torch.no_grad():
        logit = model(**inputs).logits.item()
        prob = torch.sigmoid(torch.tensor(logit)).item()

    pred = "MATCH" if logit > MATCH_THRESHOLD else "MISMATCH"
    truth = "MATCH" if true_label == 1 else "MISMATCH"
    color = 'green' if pred == truth else 'red'

    ax.imshow(img)
    ax.set_title(f'"{text}"\nPred: {pred} | True: {truth}\n(logit={logit:.2f}, prob={prob:.2f})',
                 fontsize=8, color=color)
    ax.axis('off')

plt.suptitle("ViLT Retrieval Demo", fontsize=13)
plt.tight_layout()
plt.show()

## 5. 结果总结

| 维度 | 架构解析 (Mini-ViLT) | 预训练推理 (HuggingFace) |
|-----|---------------------|------------------------|
| 参数量 | ~50K（缩小版） | ~87M（ViLT-B/32 retrieval checkpoint） |
| 使用方式 | 仅验证数据流与公式对应 | 直接输出 image-text retrieval score |
| 训练需求 | 无 | 无（本 notebook 不做微调） |
| 目的 | 理解 ViLT 每个组件的作用和张量变换 | 演示真实预训练模型的正确调用方式 |
| 主要限制 | 教学版，不追求真实精度 | 在合成几何图形数据上存在 domain gap |

In [ ]:
# ── 匹配分数分布 ──
pos_scores = test_probs[test_labels == 1].numpy()
neg_scores = test_probs[test_labels == 0].numpy()

plt.figure(figsize=(8, 4))
plt.hist(pos_scores, bins=10, alpha=0.7, label="Matched pairs")
plt.hist(neg_scores, bins=10, alpha=0.7, label="Mismatched pairs")
plt.xlabel("Predicted match probability")
plt.ylabel("Count")
plt.title("ViLT Retrieval Score Distribution")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Appendix A: 注意力可视化

使用预训练 ViLT 的注意力权重，观察模型如何在文本 token 和图像 patch 之间建立跨模态关联。

In [ ]:
# ── 提取注意力权重 ──
sample_img = all_images[0]
sample_text = all_texts[0]

inputs = processor(images=sample_img, text=sample_text, return_tensors="pt").to(device)

model.eval()
with torch.no_grad():
    outputs = model.vilt(**inputs, output_attentions=True)

# outputs.attentions: tuple of (batch, num_heads, seq_len, seq_len), one per layer
# 取最后一层的注意力权重，对所有 head 取平均
last_attn = outputs.attentions[-1].squeeze(0).mean(dim=0)  # (seq_len, seq_len)

# ViLT 序列布局: [text tokens] + [image CLS] + [image patches]
text_len = inputs['input_ids'].shape[1]
image_cls_pos = text_len
patch_start = text_len + 1
patch_count = last_attn.shape[0] - patch_start

# 提取文本 token 指向图像 patch 的跨模态注意力（排除 image CLS）
cross_attn = last_attn[:text_len, patch_start:].cpu().numpy()  # (text_len, patch_count)

# 获取 text tokens 的实际文字
tokens = processor.tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].cpu())

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(cross_attn, cmap='viridis', aspect='auto')
ax.set_yticks(range(len(tokens)))
ax.set_yticklabels(tokens, fontsize=8)
ax.set_xlabel(f"Image Patches (N={patch_count})")
ax.set_ylabel("Text Tokens")
ax.set_title(f'Cross-modal Attention: "{sample_text}" (last layer avg)')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

print(f"Text tokens: {tokens}")
print(f"Image CLS position: {image_cls_pos}")
print(f"Image patches: {patch_count}")

## Appendix B: 面试拓展

### 高频面试题

**Q1: ViLT 为什么能比传统 VLP 模型快数十倍？**

核心在于**视觉嵌入方式的简化**：
- 传统方法（UNITER）使用 Faster R-CNN 提取区域特征，耗时 ~810ms
- ViLT 使用线性投影（Patch Projection），耗时仅 ~0.4ms
- 模态交互模块（Transformer）耗时相同（~15ms），瓶颈完全在视觉编码
- ViLT 将计算重心从“分别理解”转移到“交叉理解”

**Q2: ViLT 使用单流融合（Single-Stream）而非双流融合（Dual-Stream），有什么优劣？**

| | 单流融合 (ViLT) | 双流融合 (ViLBERT) |
|--|---|---|
| 优势 | 文本和视觉 token 从第一层就交互，融合更充分 | 各模态可先独立建模，再跨模态融合 |
| 劣势 | 序列更长（$L+N$），Self-Attention 复杂度更高 | 需要额外的 Cross-Attention 模块 |
| 适用 | 需要深度多模态理解的任务（VQA、ITM） | 需要独立模态表示的任务（检索） |

**Q3: 为什么 ViLT 使用 Pre-norm 而非 Post-norm？**

- **Pre-norm**（ViT 风格）：`LN → Attention → Residual → LN → FFN → Residual`
- **Post-norm**（BERT 风格）：`Attention → Residual → LN → FFN → Residual → LN`
- Pre-norm 的梯度更直接地流向输入（残差分支不经过 LN），训练更稳定
- 对于深层 Transformer（12层+），Pre-norm 能更好地避免梯度消失/爆炸
- ViLT 继承了 ViT 的这一设计选择

**Q4: Whole Word Masking (WWM) 对 ViLT 有什么特殊意义？**

- 标准 MLM 对 subword token 随机 mask，模型可能通过语言上下文“作弊”还原 token
- 例如 `giraffe -> ["gi", "##raf", "##fe"]`，若只 mask 中间片段，模型可仅凭文本上下文猜测
- WWM 强制 mask 整个词，迫使模型必须“看”图像来还原文本
- 这对 ViLT 尤为重要，因为 ViLT 的视觉编码器较轻，需要通过 MLM 引导模型利用视觉信息

**Q5: ViLT 和 CLIP 的核心区别是什么？**

| 对比维度 | ViLT | CLIP |
|---------|------|------|
| 架构 | 单流 Transformer，共享参数 | 双塔独立编码器 |
| 融合方式 | Early fusion（token 拼接 + Self-Attention） | Late fusion（仅在输出层做对比） |
| 预训练目标 | ITM + MLM + WPA | 对比学习（InfoNCE） |
| 推理方式 | 必须同时输入图文对 | 图文可分别编码后检索 |
| 优势场景 | 深度理解任务（VQA） | 检索、零样本分类 |
| 视觉编码器 | 无 CNN（Patch Projection） | 完整 ViT 或 ResNet |

**Q6: Word Patch Alignment (WPA) 损失的作用是什么？**

- WPA 计算文本 token 和图像 patch 之间的**最优传输距离**
- 使用 IPOT（Inexact Proximal point method for Optimal Transport）算法
- 作为 ITM 的辅助损失，促进细粒度的图文对齐
- 让模型不仅学会全局匹配（ITM），还学会局部对齐（哪个词对应哪个区域）

**Q7: ViLT 的视觉嵌入没有 CNN 预处理，表达能力会不会不足？**

- 确实存在这个 trade-off：ViLT 在某些需要精细视觉理解的任务上弱于 Region-based 方法
- 但 ViLT 证明了：通过足够深的多模态 Transformer（12层），模型可以从原始 patch 学到有效的视觉表示
- 关键洞察：VLP 的瓶颈不在视觉编码器的强弱，而在多模态交互的深度
- 后续工作（如 METER、BEiT-3）在此基础上找到了更好的平衡点

**Q8: ViLT 如何处理不同尺寸的图像？**

- 训练时：图像短边缩放到 384，长边限制在 640，然后按 patch_size=32 切分
- 不同图像产生不同数量的 patches（如 384×640 -> 12×20 = 240 patches）
- 位置嵌入使用可学习参数，通过**插值**适配不同分辨率（继承自 ViT）
- 同一 batch 内用 `pixel_mask` 标记有效区域

### 延伸阅读与对比

| 对比维度 | ViLT | CLIP | ViLBERT | METER |
|---------|------|------|---------|-------|
| 视觉编码 | Linear Projection | ViT / ResNet | Faster R-CNN | ViT (CLIP 预训练) |
| 文本编码 | BERT (共享) | Transformer | BERT | RoBERTa |
| 模态交互 | Single-Stream Transformer | 对比学习（无交互） | Cross-Attention | Cross-Attention |
| 推理速度 | ~15ms | ~20ms | ~100ms | ~30ms |
| VQA 精度 | 70.6 (test-dev) | — | 72.5 | 77.7 |
| 设计重心 | 效率优先 | 通用性优先 | 精度优先 | 均衡 |

### 进阶探索方向

- **METER**：在 ViLT 基础上引入 CLIP 预训练的 ViT 作为视觉编码器，兼顾效率和精度
- **BEiT-3**：统一的多模态预训练框架，将视觉和语言都视为“语言”
- **CoCa**：结合对比学习和 captioning，同时获得 CLIP 的检索能力和 ViLT 的理解能力
- **图像增强策略**：ViLT 首次在 VLP 中系统使用图像增强，后续工作验证了其重要性